# Aşama 2: Veri Temizleme, Döngü Ayrıştırma, SOC Doğrulama ve Görselleştirme

Bu çalışma defteri:
1. Stanford `Cycling_1` (`W4`, `W8`, `W9`, `W10`) verilerindeki `Step 0 / 0 V` başlangıç artifaktlarını temizler.
2. `W10` ham Arbin Excel (`Channel_6.2.xlsx`) dosyasından gerçek hücre yüzey sıcaklığını (`Aux_Temperature`) çıkarır.
3. `Step_Index` geçişleriyle şarj/dinlenme/deşarj döngülerini ayrıştırır.
4. C/20 tanı eğrisinde kayıtlı ve Coulomb sayımıyla hesaplanan SOC'yi doğrular.
5. NASA 18650 verilerindeki `NaN` ve tekil gerilim sıçramalarını temizler ve birimli grafikleri üretir.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import run_phase2_pipeline

res = run_phase2_pipeline(
    raw_dir=PROJECT_ROOT / 'data' / 'raw',
    processed_dir=PROJECT_ROOT / 'data' / 'processed',
    tables_dir=PROJECT_ROOT / 'results' / 'tables',
    figures_dir=PROJECT_ROOT / 'results' / 'figures',
)
res['cleaning_report']

## Üretilen Doğrulama Grafikleri

In [ ]:
for fig_path in res['saved_figures']:
    print(fig_path)
    display(Image(filename=fig_path))

## Model Kalibrasyonuna Uygunluk Değerlendirmesi
- **OCV(SOC) ve Kapasite ($Q$):** `capacity_test.mat` (`Diag_1`) C/20 eğrilerinde kaydedilen ve Coulomb sayımıyla hesaplanan SOC arası fark $< 0.005$ düzeyindedir; PCHIP/spline OCV fonksiyonu için doğrudan uygundur.
- **Elektriksel Dinamikler ($R_0, R_1, C_1$):** `HPPC_test.mat` (1 Hz) ve `Cycling_1` temiz şarj profilleri yüksek çözünürlüklüdür.
- **Termal Parametreler ($R_{\text{th}}, C_{\text{th}}$):** `W10_Channel_6.2.xlsx` 3C şarj + 30 dk dinlenme boyunca gerçek yüzey sıcaklığı artışı ve soğumasını içerir.
- **Veri Ayrımı:** `W4, W8, W9, W10` kalibrasyon; `W5, V4` bağımsız doğrulama; NASA `B0005, B0029, B0053` farklı kimya/sıcaklık nitel kontrolü için ayrılmıştır.